# Import libraries

In [1]:
import locale         # To handle locale-related information
import os             # To interact with the operating system
import pickle         # For serializing and de-serializing Python object structures
import re             # To perform regular expression operations
import requests       # To send HTTP requests easily
import shutil         # For high-level file operations
import textwrap       # For text wrapping and filling
import torch          # Core library for PyTorch
import warnings       # To control warning messages
import pandas as pd

from bs4 import BeautifulSoup  # For parsing HTML and XML documents
from dotenv import load_dotenv  # To load environment variables from .env files
from huggingface_hub import notebook_login  # To log into Huggingface Hub from notebooks
from IPython.display import display, HTML  # To output HTML in Python notebooks
from langchain import HuggingFacePipeline  # Importing LangChain's HuggingFacePipeline
from langchain.chains import RetrievalQA  # For setting up a retrieval-based QA system
from langchain.document_loaders import HuggingFaceDatasetLoader, UnstructuredURLLoader  # For loading datasets
from langchain.embeddings import HuggingFaceEmbeddings  # For working with HuggingFace embeddings
from langchain.llms import HuggingFacePipeline  # LangChain's wrapper for HuggingFace's pipelines
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter  # For splitting text
from langchain.vectorstores import FAISS, Pinecone, Weaviate  # For using different vector storage solutions
from langchain_core.output_parsers import StrOutputParser  # For parsing outputs in LangChain
from langchain_core.runnables import RunnablePassthrough  # For creating runnable actions in LangChain
from langchain.prompts import PromptTemplate
from pandas import DataFrame, read_csv  # For data manipulation and analysis
from pinecone import pinecone  # Import Pinecone module for vector database operations
from pygments.lexers import PythonLexer  # For Python syntax highlighting
from pygments import highlight  # To highlight code with Pygments
from pygments.formatters import HtmlFormatter  # To format highlighted code into HTML
from rouge import Rouge  # For evaluation of summarization with ROUGE
from tabulate import tabulate  # To print tabular data
from tqdm import tqdm  # For displaying progress bars
from transformers import (AutoModelForCausalLM, AutoModelForQuestionAnswering,
                          AutoTokenizer, BitsAndBytesConfig, pipeline)  # From Hugging Face's Transformers

# Ignore all warnings
warnings.filterwarnings("ignore")

# Fixing unicode error in Google Colab
locale.getpreferredencoding = lambda: "UTF-8"


## Configuration

In [55]:
class Args():
    def __init__(self):
        # load train dataset
        self.dataset_dir = r"D:\Yeshiva\Spring24\AI\katzbot"
        
        self.train_path = r"dataset_katzbot/New_Train_QA_Pairs.csv"
        self.input_file_path = os.path.join(self.dataset_dir, self.train_path)
        
        # load test dataset
        self.test_path = r"dataset_katzbot/Test_QA_Pairs.csv"
        self.test_data = os.path.join(self.dataset_dir, self.test_path)
        self.save_text_file = "genrated_llm_rag_response.txt"
        self.predict_file = "Test_QA_Pairs_llm_rag.csv"
        self.load_data = "loaded_data_train.pkl"
        self.clf_data = "clf_dataset.csv"
        
        self.original_text = "answer"
        self.llm_response = "Cleaned_LLM_Response"
        self.rag_response = "Cleaned_RAG_Response"
        

        
        # Sets the directory to which the trained model will be saved
        self.output_dir = 'output_4'
        self.model_name = "llm_model.pt"
        self.retrieve_data = 'retriever_store.pkl'
        self.retriever_index = 'retriever_index.faiss'


args = Args()
# If the directory does not exist, it is created
if not os.path.exists(args.output_dir):
    os.makedirs(args.output_dir)

In [3]:
# Load environment variables from .env file
load_dotenv()

# Retrieve the OPENAI_API_KEY
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# print(OPENAI_API_KEY)

HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
WEAVIATE_CLUSTER = os.getenv("WEAVIATE_CLUSTER")



## Load the data from the website

In [4]:
# Step 1: Retrieve and process sitemap.xml to extract URLs
sitemap_url = "https://www.yu.edu/sitemap.xml"
sitemap_response = requests.get(sitemap_url)
soup = BeautifulSoup(sitemap_response.content, "xml")
URLs_2 = [loc.text for loc in soup.find_all("loc")]


In [5]:
len(URLs_2)

1262

In [6]:
URLs_3 = URLs_2[:10] + ["https://www.yu.edu/katz/ai"] + ["https://www.yu.edu/katz/ai/curriculum"]
URLs_3

['https://www.yu.edu/',
 'https://www.yu.edu/president/photos',
 'https://www.yu.edu/president/pillars',
 'https://www.yu.edu/admissions/events/sarachek',
 'https://www.yu.edu/admissions/events/yunmun',
 'https://www.yu.edu/admissions/events/yunmun/director',
 'https://www.yu.edu/admissions/events/yunmun/delegates',
 'https://www.yu.edu/admissions/events/yunmun/registration',
 'https://www.yu.edu/admissions/events/yunmun/handbooks',
 'https://www.yu.edu/admissions/events/yunmun/topics',
 'https://www.yu.edu/katz/ai',
 'https://www.yu.edu/katz/ai/curriculum']

In [7]:
URLs_3 = URLs_2 \
        + ["https://raw.githubusercontent.com/sahilkumar15/Research_Work/main/New_Train_QA_Pairs.csv"] \
        # + ["https://raw.githubusercontent.com/sahilkumar15/Research_Work/main/Test_QA_Pairs.csv"]
len(URLs_3)

1263

In [8]:
# load_data path
load_data_path = os.path.join(args.output_dir, args.load_data)


In [9]:

# Save your scraped data
try:
    loader = UnstructuredURLLoader(urls=URLs_3)
    data = loader.load()
    print("Number of documents loaded:", len(data))

    # Save using pickle for complex data structures
    with open(load_data_path, 'wb') as f:
        pickle.dump(data, f)
    print("Data saved successfully.")
except Exception as e:
    print("Failed to load and save data:", str(e))


Following dependencies are missing: pikepdf. Please install them using `pip install pikepdf`.
PDF text extraction failed, skip text extraction...
Error fetching or processing https://www.yu.edu/sites/default/files/inline-files/Dual_Majors_and_Minors_at_Sy_Syms.pdf, exception: unstructured_inference is not installed, pytesseract is not installed and the text of the PDF is not extractable. To process this file, install unstructured_inference, install pytesseract, or remove copy protection from the PDF.


Number of documents loaded: 1262
Data saved successfully.


In [10]:

# Loading data from a pickle file
try:
    with open(load_data_path, 'rb') as f:
        data = pickle.load(f)
    print("Data loaded successfully.")
    # Now loaded_data contains the data that was previously saved
except Exception as e:
    print("Failed to load data:", str(e))

Data loaded successfully.


## Text Splitting

In [11]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
docs = text_splitter.split_documents(data)

## Embedding Convertion

In [12]:
# embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
# model_kwargs = {"device": "cpu"}
# embeddings = HuggingFaceEmbeddings(
#     model_name=embedding_model_name, model_kwargs=model_kwargs
# )

In [13]:
# import torch
print(torch.__version__)
print("CUDA Available:", torch.cuda.is_available())


2.3.0+cu121
CUDA Available: True


In [14]:
# Define the path to the pre-trained model you want to use
modelPath = "sentence-transformers/all-MiniLM-l6-v2"

# Create a dictionary with model configuration options, specifying to use the GPU for computations
model_kwargs = {'device': 'cuda'}

# Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
encode_kwargs = {'normalize_embeddings': False}

# Initialize an instance of HuggingFaceEmbeddings with the specified parameters
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,     # Provide the pre-trained model's path
    model_kwargs=model_kwargs, # Pass the model configuration options
    
    encode_kwargs=encode_kwargs # Pass the encoding options
)


In [15]:
text = "This is a test document."
query_result = embeddings.embed_query(text)
query_result[:3]

[-0.03833853825926781, 0.12346473336219788, -0.02864292822778225]

## Vector Database Storage

In [16]:
print(docs[:2]) 

[Document(page_content='Skip page description and go main skip links. First time visitors should read the description.\n\nThis is the Yeshiva University website home page.\n    This page includes, from top to bottom:\n    1. A large image of New York City with a short navigation menu on top which slides up when scrolling down the page.\n    2. Utility and main navigation links.\n    3. A section of four featured topics.\n    4. A section on acedemics with links to programs.\n    5. A news section with four featured news items.\n    6. An events section with four featured events.\n    7. A section with six major navigation links.\n    8. The page footer with various links.\n    Skip link to navigation and search is next, followed by skip link to main content.\n\nSkip to main navigation and search\n\nHome Page Cover', metadata={'source': 'https://www.yu.edu/'}), Document(page_content='Home Page Cover\n\nWhat follows is an image of New York City at night with the Yeshiva University logo o

In [17]:
db = FAISS.from_documents(docs, embeddings)

In [18]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Load quantized model

In [19]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

## Setup the LLM chain

In [20]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=400,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>

 """
 
 # Simplified prompt template focusing only on the question
# prompt_template = """
# {question}
# """


prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()

In [21]:
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

## Compare the results

In [22]:
question = "who is RABBI DR. ARI BERMAN?"


In [23]:
llm_response = llm_chain.invoke({"context": "", "question": question})
llm_response

"\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n\n\n</s>\n<|user|>\nwho is RABBI DR. ARI BERMAN?\n</s>\n<|assistant|>\n\n  Rabbi Dr. Ari Berman is a prominent Jewish religious leader and academic with multiple credentials in both fields. He holds a doctorate degree in Jewish Studies from the University of Oxford, as well as rabbinical ordination from Yeshiva University's Rabbi Isaac Elchanan Theological Seminary (RIETS). In addition to his academic achievements, Rabbi Berman serves as the Senior Rabbi of London's West End Synagogue, one of the most prestigious Orthodox synagogues in Europe. His expertise in Jewish studies and leadership in the Jewish community has earned him recognition as an influential figure in contemporary Judaism."

In [24]:
print(llm_response)


<|system|>
Answer the question based on your knowledge. Use the following context to help:



</s>
<|user|>
who is RABBI DR. ARI BERMAN?
</s>
<|assistant|>

  Rabbi Dr. Ari Berman is a prominent Jewish religious leader and academic with multiple credentials in both fields. He holds a doctorate degree in Jewish Studies from the University of Oxford, as well as rabbinical ordination from Yeshiva University's Rabbi Isaac Elchanan Theological Seminary (RIETS). In addition to his academic achievements, Rabbi Berman serves as the Senior Rabbi of London's West End Synagogue, one of the most prestigious Orthodox synagogues in Europe. His expertise in Jewish studies and leadership in the Jewish community has earned him recognition as an influential figure in contemporary Judaism.


In [25]:
rag_response = rag_chain.invoke(question)
rag_response

'\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n[Document(page_content=\'Rabbi Dr. Ari Berman\\n      \\n                  \\n                  \\n        \\n                  \\n\\n            Rabbi Dr. Ari Berman\\n      \\n                \\n      \\n          \\n\\n\\n                  \\n\\n\\n        \\n      \\n        \\n\\n                \\n                  \\n              \\n                  \\n\\n  \\n\\n\\n      \\n      \\n              \\n                      \\n\\n            Rabbi Elchanan Adler\\n      \\n                      \\n          \\n                      \\n\\n            Read more about Rabbi Adler\', metadata={\'source\': \'https://www.yu.edu/riets/faculty\'}), Document(page_content="Skip to main content\\n\\nSkip to search\\n\\nApply       \\n          \\n    \\n                Graduate\\n\\nGive\\n\\nLife\\n\\nOffices\\n                    \\n          \\n    \\n                \\n      Academics\\n  

In [26]:
print(rag_response)


<|system|>
Answer the question based on your knowledge. Use the following context to help:

[Document(page_content='Rabbi Dr. Ari Berman\n      \n                  \n                  \n        \n                  \n\n            Rabbi Dr. Ari Berman\n      \n                \n      \n          \n\n\n                  \n\n\n        \n      \n        \n\n                \n                  \n              \n                  \n\n  \n\n\n      \n      \n              \n                      \n\n            Rabbi Elchanan Adler\n      \n                      \n          \n                      \n\n            Read more about Rabbi Adler', metadata={'source': 'https://www.yu.edu/riets/faculty'}), Document(page_content="Skip to main content\n\nSkip to search\n\nApply       \n          \n    \n                Graduate\n\nGive\n\nLife\n\nOffices\n                    \n          \n    \n                \n      Academics\n            \n                \n      Alumni\n            \n            

In [27]:


def remove_preceding_text(text, start_phrase):
    """
    This function removes all text before and including the specified start_phrase in the given text.
    It then trims leading and trailing whitespace from the remaining text.
    If the start_phrase is not found, it returns the original text trimmed for whitespace.
    """
    # The pattern finds the specified phrase and captures everything after it
    pattern = re.compile(re.escape(start_phrase) + r'(.*)', re.DOTALL)
    match = pattern.search(text)
    # Use strip() to remove leading and trailing whitespace from the remaining text
    return match.group(1).strip() if match else text.strip()  # Returns the trimmed text after the phrase

cleaned_text = remove_preceding_text(rag_response, "<|assistant|>")
print(cleaned_text)


RABBI DR. ARI BERMAN is the current president of Yeshiva University, a prominent Jewish institution of higher learning. He assumed this position in 2017 after being invested as the university's fifth president. Prior to becoming president, Rabbi Berman served as the dean of the Rabbi Isaac Elchanan Theological Seminary (RIETS) at Yeshiva University from 2006 to 2013. In addition to his administrative roles, Rabbi Berman is also a respected scholar and author, having earned a Ph.D. In Jewish history from the Jewish Theological Seminary and published several books and articles on Jewish thought and history. As president, Rabbi Berman has focused on strengthening Yeshiva University's commitment to its core Torah values while expanding its academic offerings and increasing enrollment, philanthropy, and academic rankings. He is also an active and influential voice within the Jewish community, frequently delivering lectures and speaking out on issues related to Jewish education, leadership, 

In [28]:
cleaned_text = remove_preceding_text(llm_response, "<|assistant|>")
print(cleaned_text)

Rabbi Dr. Ari Berman is a prominent Jewish religious leader and academic with multiple credentials in both fields. He holds a doctorate degree in Jewish Studies from the University of Oxford, as well as rabbinical ordination from Yeshiva University's Rabbi Isaac Elchanan Theological Seminary (RIETS). In addition to his academic achievements, Rabbi Berman serves as the Senior Rabbi of London's West End Synagogue, one of the most prestigious Orthodox synagogues in Europe. His expertise in Jewish studies and leadership in the Jewish community has earned him recognition as an influential figure in contemporary Judaism.


In [29]:
question = "What is the curriculum in M.S. in Artificial Intelligence?"
llm_response = llm_chain.invoke({"context": "", "question": question})
rag_response = rag_chain.invoke(question)

rag_cleaned_text = remove_preceding_text(rag_response, "<|assistant|>")
print(rag_cleaned_text)

print("\n\n")
llm_cleaned_text = remove_preceding_text(llm_response, "<|assistant|>")
print(llm_cleaned_text)

The curriculum for the M.S. in Artificial Intelligence program includes both core requirements and elective courses. Here is a breakdown of the program:

  1. Core Requirements (24 credits):
    - Data Acquisition and Management
    - Computational Statistics and Probability
    - Numerical Methods
    - Predictive Models
    - Machine Learning
    - Artificial Intelligence
    - Neural Networks and Deep Learning
    - AI Capstone: R&D Experience

  2. Electives (12 credits):
    - Bayesian Methods
    - AI Product Studio
    - Natural Language Processing
    - Data Visualization
    - Advanced Data Engineering
    - Complex Systems: Financial Time Series Analysis
    - Special Topics (1-3 credits)
    - Independent Study (1-3 credits)
    - Internship (1-3 credits)*

  The program aims to provide students with functional knowledge of Artificial Intelligence principles, practices, workflows, technology, and tools, including supervised learning, unsupervised learning, and transfer learn

## Evaluation

In [30]:
import pandas as pd
test_df = pd.read_csv(args.test_data)
# test_df = test_df.iloc[:10]
print(test_df.shape)
test_df.head()

(2081, 2)


,question,answer
0,What is the student/faculty ratio at this univ...,The student/faculty ratio at Yeshiva Universit...
1,How often do students get to interact with pro...,Students at Yeshiva University have ample oppo...
2,How active is the student community on campus?,The student community at Yeshiva University is...
3,Can you tell me more about the extracurricular...,Yeshiva University offers a wide range of extr...
4,What percentage of students get financial aid?,Approximately 85% of students at Yeshiva Unive...


In [34]:
import time
import pandas as pd
import re

def format_time(seconds):
    """
    Converts a given time in seconds to a formatted string displaying hours, minutes, and seconds.
    """
    # divmod returns a tuple (quotient, remainder)
    hours, remainder = divmod(seconds, 3600)  # Divide seconds by 3600 to get hours and remainder
    minutes, seconds = divmod(remainder, 60)  # Divide remainder by 60 to get minutes and seconds
    return f"{int(hours)}:{int(minutes)}:{seconds:.2f}"


def remove_preceding_text(text, start_phrase = "<|assistant|>"):
    """
    Removes all text before and including the specified start_phrase in the given text.
    Then trims leading and trailing whitespace from the remaining text.
    If the start_phrase is not found, it returns the original text trimmed for whitespace.
    """
    pattern = re.compile(re.escape(start_phrase) + r'(.*)', re.DOTALL)
    match = pattern.search(text)
    return match.group(1).strip() if match else text.strip()

def process_questions(dataframe, llm_chain, rag_chain):
    """
    Processes a dataframe of questions, invokes two different response systems,
    cleans the responses, and appends them to the dataframe.
    """
    start_time = time.time()  # Start timing
    
    # Add columns to store the responses
    dataframe['Cleaned_LLM_Response'] = ''
    dataframe['Cleaned_RAG_Response'] = ''
    
    # df = tqdm(dataframe.iterrows(), total=len(dataframe)):
    for index, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        question = row['question']
        
        # Simulating responses from two systems
        llm_response = llm_chain.invoke({"context": "", "question": question})
        rag_response = rag_chain.invoke(question)
        
        # Cleaning the responses
        cleaned_llm = remove_preceding_text(llm_response)
        cleaned_rag = remove_preceding_text(rag_response)
        
        # Storing responses in the dataframe
        dataframe.at[index, 'Cleaned_LLM_Response'] = cleaned_llm
        dataframe.at[index, 'Cleaned_RAG_Response'] = cleaned_rag
    
    end_time = time.time()  # End timing
    total_time = end_time - start_time
    
     # Convert and print the formatted time
    formatted_time = format_time(total_time)
    print(f"Total time taken: {formatted_time}")
    
    return dataframe

# Example usage
# Assuming test_df is a DataFrame and llm_chain and rag_chain are defined elsewhere
processed_df = process_questions(test_df, llm_chain, rag_chain)
processed_df.head()


Total time taken: 10:44:57.14


,question,answer,Cleaned_LLM_Response,Cleaned_RAG_Response
0,What is the student/faculty ratio at this univ...,The student/faculty ratio at Yeshiva Universit...,I do not have information about a specific uni...,"According to the provided context, the student..."
1,How often do students get to interact with pro...,Students at Yeshiva University have ample oppo...,The frequency of interactions between students...,"Based on the provided context, students have t..."
2,How active is the student community on campus?,The student community at Yeshiva University is...,"I do not have firsthand experience, but based ...","Based on the provided context, it can be infer..."
3,Can you tell me more about the extracurricular...,Yeshiva University offers a wide range of extr...,I do not have information about a specific uni...,"Based on the information provided, it seems th..."
4,What percentage of students get financial aid?,Approximately 85% of students at Yeshiva Unive...,The percentage of students who receive financi...,Over 80% of Yeshiva University students receiv...


In [35]:
# import time
# import pandas as pd
# import re

# def format_time(seconds):
#     """
#     Converts a given time in seconds to a formatted string displaying hours, minutes, and seconds.
#     """
#     # divmod returns a tuple (quotient, remainder)
#     hours, remainder = divmod(seconds, 3600)  # Divide seconds by 3600 to get hours and remainder
#     minutes, seconds = divmod(remainder, 60)  # Divide remainder by 60 to get minutes and seconds
#     return f"{int(hours)} : {int(minutes)} : {seconds:.2f} "


# def remove_preceding_text(text, start_phrase = "<|assistant|>"):
#     """
#     Removes all text before and including the specified start_phrase in the given text.
#     Then trims leading and trailing whitespace from the remaining text.
#     If the start_phrase is not found, it returns the original text trimmed for whitespace.
#     """
#     pattern = re.compile(re.escape(start_phrase) + r'(.*)', re.DOTALL)
#     match = pattern.search(text)
#     return match.group(1).strip() if match else text.strip()

# def process_questions(dataframe, llm_chain, rag_chain, batch_size=10):
#     start_time = time.time()
#     responses_llm = []
#     responses_rag = []

#     # Process in batches
#     for start in range(0, len(dataframe), batch_size):
#         end = start + batch_size
#         questions_batch = dataframe.iloc[start:end]['question'].tolist()
        
#         # Batch invoke
#         responses_llm.extend(llm_chain.invoke_batch({"context": [""] * len(questions_batch), "question": questions_batch}))
#         responses_rag.extend(rag_chain.invoke_batch(questions_batch))

#     # Clean responses
#     dataframe['Cleaned_LLM_Response'] = [remove_preceding_text(resp) for resp in responses_llm]
#     dataframe['Cleaned_RAG_Response'] = [remove_preceding_text(resp) for resp in responses_rag]

    
#     end_time = time.time()  # End timing
#     total_time = end_time - start_time
    
#      # Convert and print the formatted time
#     formatted_time = format_time(total_time)
#     print(f"Total time taken: {formatted_time}")
    
#     return dataframe

# # Example usage
# # Assuming test_df is a DataFrame and llm_chain and rag_chain are defined elsewhere
# processed_df = process_questions(test_df, llm_chain, rag_chain, batch_size=10)
# processed_df.head()


In [36]:
# Specify the file path where you want to save the updated DataFrame
test_predict = os.path.join(args.output_dir, args.predict_file)

# Save the updated dataframe to a new CSV file
processed_df.to_csv(test_predict, index=False)

In [43]:
text_file = os.path.join(args.output_dir, args.save_text_file)

with open(text_file, 'w') as f:
    for i, row in tqdm(processed_df.iterrows(), total=len(test_df)):
        question, answer, Cleaned_LLM_Response,  Cleaned_RAG_Response = row
        f.write(f"Question {i+1}/{len(test_df)} => {question}\n")
        f.write(f"Answer => {answer}\n")
        f.write(f"LLM_Response => {Cleaned_LLM_Response}\n")
        f.write(f"RAG_Response => {Cleaned_RAG_Response}\n")
        # f.write('='*100)
        f.write('\n\n')

# Optionally, you can print a message indicating that the output has been saved
print(f"Output saved to {text_file}")

100%|██████████| 2081/2081 [00:00<00:00, 50471.83it/s]

Output saved to output_4\genrated_llm_rag_response.txt


In [38]:
def calculate_rouge_scores(preds, original_text_col, predicted_result_col):
    rouge_scores = []
    hyps = []  # list of predicted answers
    refs = []  # list of reference answers

    for index, row in preds.iterrows():
        act_ans = str(row[original_text_col])
        pred_ans = str(row[predicted_result_col])

        hyps.append(pred_ans)
        refs.append(act_ans)

    rouge = Rouge()
    scores = rouge.get_scores(hyps, refs, avg=True)
    rouge_scores.append(scores)

    # Create a list of dictionaries for tabulating the scores
    score_table = [{'Metric': metric, 'Precision': score['p'], 'Recall': score['r'], 'F1-Score': score['f']} for metric, score in scores.items()]
    metrics_df = pd.DataFrame(score_table)
    
    return metrics_df

In [40]:
metrics_df = calculate_rouge_scores(processed_df, args.original_text, args.rag_response)
metrics_df

,Metric,Precision,Recall,F1-Score
0,rouge-1,0.293373,0.571497,0.367401
1,rouge-2,0.182687,0.359168,0.226178
2,rouge-l,0.276207,0.531375,0.344360


---